In [0]:
# MAGIC %md
# MAGIC # PySpark Data Processing Pipeline: NYC Taxi Data
# MAGIC 
# MAGIC This notebook implements a full data processing pipeline using PySpark on the built-in NYC Taxi dataset.
# MAGIC ---

In [0]:
# MAGIC %md
# MAGIC ## Step 1: Setup and Data Loading
# MAGIC 
# MAGIC We'll load the main taxi trip data (yellow_tripdata) and the taxi zone lookup table. These are available in the Databricks sample datasets.

In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, round, avg, count, sum, lit, to_timestamp

trip_data_path = "/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-01.csv.gz"
zone_lookup_path = "/databricks-datasets/nyctaxi/taxizone/taxi_zone_lookup.csv"

# Output path for our processed data
output_path = "/tmp/NYCTaxiPipeline/analysis_results/"

In [0]:
# Load Trip Data
# We'll let Spark infer the schema and handle the header
raw_trips_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(trip_data_path)

zone_lookup_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(zone_lookup_path)

print("Data loaded successfully.")

Data loaded successfully.


In [0]:
# MAGIC %md
# MAGIC ## Step 2: Data Processing Pipeline (DataFrame API)
# MAGIC 
# MAGIC We will now apply a series of transformations.
# MAGIC 
# MAGIC **Optimization:** We apply filters **as early as possible** to reduce the amount of data processed in later stages.

In [0]:
# Transformation 1: Apply Early Filters (2+ filters)
# Filter out nulls and trips with no passengers or zero distance
filtered_trips_df = raw_trips_df \
    .filter(col("passenger_count") > 0) \
    .filter(col("trip_distance") > 0) \
    .filter(col("PULocationID").isNotNull()) \
    .filter(col("DOLocationID").isNotNull())

In [0]:
# Transformation 2: Column Transformations (withColumn)
# Cast timestamps and create new features
transformed_trips_df = filtered_trips_df \
    .withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime"))) \
    .withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime"))) \
    .withColumn("trip_duration_seconds", (col("tpep_dropoff_datetime").cast("long") - col("tpep_pickup_datetime").cast("long"))) \
    .withColumn("trip_duration_minutes", round(col("trip_duration_seconds") / 60, 2)) \
    .withColumn("cost_per_mile", round(col("total_amount") / col("trip_distance"), 2))

In [0]:
# Transformation 3: Join Operation
# We create aliases for the zone_lookup_df to avoid ambiguous columns
pickup_zones = zone_lookup_df.alias("pickup")
dropoff_zones = zone_lookup_df.alias("dropoff")

# Join trips with zones to get human-readable location names
joined_df = transformed_trips_df \
    .join(pickup_zones.withColumnRenamed("Zone", "Pickup_Zone").withColumnRenamed("Borough", "Pickup_Borough"),
          col("PULocationID") == col("pickup.LocationID"), # Use aliased column
          "left") \
    .join(dropoff_zones.withColumnRenamed("Zone", "Dropoff_Zone").withColumnRenamed("Borough", "Dropoff_Borough"),
          col("DOLocationID") == col("dropoff.LocationID"), # Use aliased column
          "left")

# Prune columns to keep only what we need (another optimization)
analysis_df = joined_df.select(
    "tpep_pickup_datetime",
    "passenger_count",
    "trip_distance",
    "total_amount",
    "payment_type",
    "trip_duration_minutes",
    "cost_per_mile",
    "Pickup_Zone",
    "Pickup_Borough",
    "Dropoff_Zone",
    "Dropoff_Borough"
)

In [0]:
# Transformation 4: GroupBy and Aggregation
# Finding the busiest boroughs
borough_summary_df = analysis_df \
    .groupBy("Pickup_Borough") \
    .agg(
        count("*").alias("total_trips"),
        avg("total_amount").alias("avg_fare"),
        avg("trip_distance").alias("avg_distance")
    ) \
    .orderBy(col("total_trips").desc())

In [0]:
# Action 1: Show the result of the aggregation
print("Borough Summary:")
borough_summary_df.show()

Borough Summary:
+--------------+-----------+------------------+------------------+
|Pickup_Borough|total_trips|          avg_fare|      avg_distance|
+--------------+-----------+------------------+------------------+
|     Manhattan|    6807339|13.507881991004613|2.2225421974723334|
|        Queens|     446545|  45.0263214010611| 11.59962187461474|
|       Unknown|     146149|15.418409978866686|2.5998101937064173|
|      Brooklyn|      82588|19.753469632409633| 4.517867850050868|
|         Bronx|      14322| 26.06881092025827| 6.943785784108348|
| Staten Island|        292| 50.37804794520529|  13.4234589041096|
|           EWR|        145| 83.89599999999992| 7.477448275862066|
+--------------+-----------+------------------+------------------+



In [0]:
# MAGIC %md
# MAGIC ## Step 3: SQL Queries
# MAGIC 
# MAGIC We'll register our main `analysis_df` as a temporary view to run SQL queries.

In [0]:
# Register the DataFrame as a temporary SQL view
analysis_df.createOrReplaceTempView("taxi_trips_view")

In [0]:
%sql
-- SQL Query 1: Average fare by payment type
SELECT
  CASE 
    WHEN payment_type = 1 THEN 'Credit Card'
    WHEN payment_type = 2 THEN 'Cash'
    WHEN payment_type = 3 THEN 'No Charge'
    WHEN payment_type = 4 THEN 'Dispute'
    ELSE 'Unknown/NA'
  END AS payment_method,
  COUNT(*) AS num_trips,
  AVG(total_amount) AS avg_fare
FROM
  workspace.default.nyc_taxi_analysis
GROUP BY
  payment_type
ORDER BY
  num_trips DESC;

payment_method,num_trips,avg_fare
Credit Card,5379690,16.562173260644716
Cash,2082221,12.59409413828835
No Charge,26158,35.858666182450335
Dispute,9311,8.913328321340762


In [0]:
%sql
-- SQL Query 2: Top 10 Busiest Pickup Zones
SELECT
  Pickup_Zone,
  Pickup_Borough,
  COUNT(*) AS total_pickups
FROM
  workspace.default.nyc_taxi_analysis
WHERE
  Pickup_Zone IS NOT NULL
GROUP BY
  Pickup_Zone,
  Pickup_Borough
ORDER BY
  total_pickups DESC
LIMIT 10;


Pickup_Zone,Pickup_Borough,total_pickups
Upper East Side South,Manhattan,327067
Upper East Side North,Manhattan,317698
Midtown Center,Manhattan,306727
Midtown East,Manhattan,272037
Times Sq/Theatre District,Manhattan,258472
Penn Station/Madison Sq West,Manhattan,254988
Clinton East,Manhattan,236249
Murray Hill,Manhattan,234551
Union Sq,Manhattan,233505
Lincoln Square East,Manhattan,230926


In [0]:
# MAGIC %md
# MAGIC ## Step 4: Performance Analysis
# MAGIC 
# MAGIC ### 4.1 `.explain()` Plan
# MAGIC 
# MAGIC Checking the physical plan for our `borough_summary_df` aggregation.

In [0]:
print("Physical Plan for Borough Summary:")
borough_summary_df.explain()

Physical Plan for Borough Summary:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonSort [total_trips#14440L DESC NULLS LAST]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#23864]
               +- PhotonShuffleExchangeSink rangepartitioning(total_trips#14440L DESC NULLS LAST, 1024)
                  +- PhotonGroupingAgg(keys=[Pickup_Borough#14425], functions=[finalmerge_count(merge count#14581L) AS count(1)#14577L, finalmerge_avg(merge sum#14584, count#14585L) AS avg(total_amount)#14578, finalmerge_avg(merge sum#14588, count#14589L) AS avg(trip_distance)#14579])
                     +- PhotonShuffleExchangeSource
                        +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#23858]
                           +- PhotonShuffleExchangeSink hashpartitioning(Pickup_Borough#14425, 1024)
                              +- PhotonGroupingAgg(key

In [0]:
# MAGIC %md
# MAGIC ### 4.2 Spark UI and Query Details
# MAGIC 

In [0]:
# Action 2: Write results to a Delta Table
# This uses saveAsTable for persistence, which is a better practice in Databricks
# and avoids low-level DBFS file path issues.

print("Writing final data to Delta Lake table 'nyc_taxi_analysis'...")

# Note: We are using a temporary table name. This will be visible in the Data Explorer.
analysis_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("Pickup_Borough") \
    .saveAsTable("nyc_taxi_analysis")

print("Write complete. Data is now saved as a partitioned Delta table named 'nyc_taxi_analysis'.")

Writing final data to Delta Lake table 'nyc_taxi_analysis'...
Write complete. Data is now saved as a partitioned Delta table named 'nyc_taxi_analysis'.


In [0]:
import time

start_time = time.time()
count_1 = analysis_df.count()
end_time = time.time()
uncached_time = end_time - start_time
print(f"Uncached count: {count_1} (Took {uncached_time:.2f} seconds)")

Uncached count: 7497380 (Took 11.75 seconds)


In [0]:
# Cache the DataFrame -- not possible on serverless setting, using alternative
display(analysis_df)

tpep_pickup_datetime,passenger_count,trip_distance,total_amount,payment_type,trip_duration_minutes,cost_per_mile,Pickup_Zone,Pickup_Borough,Dropoff_Zone,Dropoff_Borough
2019-01-01T00:46:40.000Z,1,1.5,9.95,1,6.67,6.63,Manhattan Valley,Manhattan,Upper West Side South,Manhattan
2019-01-01T00:59:47.000Z,1,2.6,16.3,1,19.2,6.27,Upper West Side South,Manhattan,West Chelsea/Hudson Yards,Manhattan
2019-01-01T00:21:28.000Z,1,1.3,9.05,1,7.15,6.96,Midtown North,Manhattan,Sutton Place/Turtle Bay North,Manhattan
2019-01-01T00:32:01.000Z,1,3.7,18.5,1,13.63,5.0,Sutton Place/Turtle Bay North,Manhattan,Astoria,Queens
2019-01-01T00:57:32.000Z,2,2.1,13.0,1,12.0,6.19,Lenox Hill West,Manhattan,Union Sq,Manhattan
2019-01-01T00:24:04.000Z,2,2.8,19.55,1,23.03,6.98,West Chelsea/Hudson Yards,Manhattan,Midtown East,Manhattan
2019-01-01T00:21:59.000Z,1,0.7,8.5,1,6.42,12.14,Upper West Side North,Manhattan,Manhattan Valley,Manhattan
2019-01-01T00:45:21.000Z,1,8.7,42.95,1,45.73,4.94,Midtown North,Manhattan,Boerum Hill,Brooklyn
2019-01-01T00:43:19.000Z,1,6.3,28.5,1,24.38,4.52,Stuy Town/Peter Cooper Village,Manhattan,Boerum Hill,Brooklyn
2019-01-01T00:58:24.000Z,1,2.7,15.3,1,16.9,5.67,Lenox Hill West,Manhattan,Union Sq,Manhattan


In [0]:
# Second run: Cached
# The .count() action will force the cache to be populated
start_time = time.time()
count_2 = analysis_df.count()
end_time = time.time()
cached_time_1 = end_time - start_time
print(f"First cached count: {count_2} (Took {cached_time_1:.2f} seconds)")

First cached count: 7497380 (Took 11.75 seconds)


In [0]:
# Third run: Fully cached
start_time = time.time()
count_3 = analysis_df.count()
end_time = time.time()
cached_time_2 = end_time - start_time
print(f"Second cached count: {count_3} (Took {cached_time_2:.2f} seconds)")

Second cached count: 7497380 (Took 11.51 seconds)


In [0]:
# MAGIC %md
# MAGIC ## Step 5: Actions vs. Transformations
# MAGIC 
# MAGIC This demonstrates Spark's lazy evaluation.

In [0]:
# 1. Define a Transformation
# This returns a new DataFrame definition but does *not* execute a job.
print("Defining a transformation...")
highly_paid_trips = analysis_df.filter(col("total_amount") > 100)

print("Transformation defined. No job was triggered.")
print("Type:", type(highly_paid_trips))

Defining a transformation...
Transformation defined. No job was triggered.
Type: <class 'pyspark.sql.connect.dataframe.DataFrame'>


In [0]:
# 2. Call an Action
# This forces Spark to execute the plan (load -> filter -> filter -> ... -> join -> filter)
print("\nCalling an action (.show())...")
highly_paid_trips.show(5)

print("Action complete. A Spark job was triggered to get this result.")


Calling an action (.show())...
+--------------------+---------------+-------------+------------+------------+---------------------+-------------+-----------+--------------+------------+---------------+
|tpep_pickup_datetime|passenger_count|trip_distance|total_amount|payment_type|trip_duration_minutes|cost_per_mile|Pickup_Zone|Pickup_Borough|Dropoff_Zone|Dropoff_Borough|
+--------------------+---------------+-------------+------------+------------+---------------------+-------------+-----------+--------------+------------+---------------+
| 2019-01-01 00:23:11|              1|         0.34|       185.3|           1|                 0.23|        545.0|         NA|       Unknown|          NA|        Unknown|
| 2019-01-01 00:38:36|              2|        33.19|      131.88|           1|                42.95|         3.97|   Gramercy|     Manhattan|          NA|        Unknown|
| 2019-01-01 00:13:17|              1|         44.1|       150.3|           2|                52.93|         3.41

In [0]:
# MAGIC %md
# MAGIC 
# MAGIC Building a simple linear regression model to predict `total_amount` based on `trip_distance`.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# Select features for the model
# We need to filter out edge cases for a stable model
ml_df = analysis_df \
    .filter(col("total_amount") > 0) \
    .filter(col("total_amount") < 500) \
    .filter(col("trip_distance") > 0) \
    .filter(col("trip_distance") < 100) \
    .select("total_amount", "trip_distance", "passenger_count") \
    .na.drop()

# 1. Assemble features into a single vector
assembler = VectorAssembler(
    inputCols=["trip_distance", "passenger_count"],
    outputCol="features"
)
ml_data = assembler.transform(ml_df)

# 2. Split data
(training_data, test_data) = ml_data.randomSplit([0.8, 0.2], seed=42)

# 3. Define and train the model
lr = LinearRegression(featuresCol="features", labelCol="total_amount")
lr_model = lr.fit(training_data)

# 4. Print model coefficients
print(f"Model Coefficients:")
print(f"Intercept: {lr_model.intercept:.2f}")
print(f"Trip Distance: {lr_model.coefficients[0]:.2f}")
print(f"Passenger Count: {lr_model.coefficients[1]:.2f}")

# 5. Evaluate on test data
predictions = lr_model.transform(test_data)
predictions.select("total_amount", "prediction").show(5)

Model Coefficients:
Intercept: 5.97
Trip Distance: 3.37
Passenger Count: -0.02
+------------+-----------------+
|total_amount|       prediction|
+------------+-----------------+
|         0.3|6.288730200177323|
|         0.3|6.288730200177323|
|         0.3|6.288730200177323|
|         0.3|6.288730200177323|
|         0.3|6.288730200177323|
+------------+-----------------+
only showing top 5 rows


In [0]:
# Clean up the FileStore directory (optional)
# dbutils.fs.rm(output_path, recurse=True)